# Notebook 28 — Embeddings and Semantic Search

    ## Learning objectives

    - Normalize embeddings and implement cosine/dot-product retrieval
- Separate bi-encoder retrieval from cross-encoder reranking
- Measure recall@k, MRR, latency, and index tradeoffs

    Cells labeled **optional GPU/remote** are deliberately guarded. Read them first,
    then opt in when the required hardware or Hugging Face Inference access is available.


In [ ]:
# Colab/local environment setup — run this cell first.
import importlib.util
import os
import platform
import subprocess
import sys

# VS Code's Colab extension can attach to a Colab kernel before `google.colab`
# has been imported, so checking only sys.modules produces a false negative.
try:
    HAS_GOOGLE_COLAB = importlib.util.find_spec("google.colab") is not None
except ModuleNotFoundError:  # The parent `google` namespace is absent locally.
    HAS_GOOGLE_COLAB = False
IN_COLAB = HAS_GOOGLE_COLAB or bool(os.getenv("COLAB_RELEASE_TAG")) or bool(os.getenv("COLAB_GPU"))
PACKAGES = ['sentence-transformers>=4,<6']

if IN_COLAB and PACKAGES:
    print("Installing notebook dependencies in the Colab runtime...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", *PACKAGES])

# Load an HF token from Colab Secrets without displaying it. In Colab, create a
# secret named HF_TOKEN (or HUGGINGFACE_TOKEN) and enable notebook access.
token = os.getenv("HF_TOKEN") or os.getenv("HUGGINGFACE_TOKEN")
if IN_COLAB and not token:
    from google.colab import userdata
    for secret_name in ("HF_TOKEN", "HUGGINGFACE_TOKEN"):
        try:
            token = userdata.get(secret_name)
        except Exception:
            token = None
        if token:
            break
elif not IN_COLAB:
    try:
        from dotenv import load_dotenv
        load_dotenv(".env")
    except ImportError:
        pass
    token = os.getenv("HF_TOKEN") or os.getenv("HUGGINGFACE_TOKEN")

# `HF_TOKEN` is the canonical huggingface_hub variable. The course also sets its
# descriptive alias because some lesson code uses HUGGINGFACE_TOKEN explicitly.
if token:
    os.environ["HF_TOKEN"] = token
    os.environ["HUGGINGFACE_TOKEN"] = token

try:
    import torch
    accelerator = torch.cuda.get_device_name(0) if torch.cuda.is_available() else (
        "Apple MPS" if getattr(torch.backends, "mps", None) and torch.backends.mps.is_available() else "CPU"
    )
    print(f"runtime={platform.platform()} | Python={platform.python_version()} | accelerator={accelerator}")
    if False and not torch.cuda.is_available():
        print("WARNING: this training notebook is designed for a Colab GPU runtime. "
              "Select Runtime > Change runtime type > T4 GPU (or better).")
except ImportError:
    print(f"runtime={platform.platform()} | Python={platform.python_version()}")

print("Colab runtime detected:", IN_COLAB)
print("Hugging Face token configured:", bool(os.getenv("HF_TOKEN")))
if IN_COLAB and not token:
    print("Add an HF_TOKEN secret in Colab, enable notebook access, then rerun this cell.")


## 28.1 Dense retrieval

A bi-encoder maps queries and documents independently into vectors. Precomputed document
vectors make retrieval fast. With unit-normalized vectors, cosine similarity equals dot
product. The embedding model's training objective determines what “similar” means;
generic semantic similarity may not match your domain's relevance.


In [ ]:
from sentence_transformers import SentenceTransformer
import numpy as np

docs = [
    "Gradient accumulation simulates a larger batch using several microbatches.",
    "RoPE rotates query and key feature pairs according to token position.",
    "FlashAttention reduces attention IO and intermediate memory.",
    "LoRA trains low-rank updates while freezing base weights.",
]
queries = ["How can I train with a bigger effective batch?", "What rotates Q and K?"]
encoder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
D = encoder.encode(docs, normalize_embeddings=True)
Q = encoder.encode(queries, normalize_embeddings=True)
scores = Q @ D.T
for query, row in zip(queries, scores):
    order = np.argsort(-row)
    print("\n", query)
    for i in order[:2]: print(f"  {row[i]:.3f} {docs[i]}")


In [ ]:
def recall_at_k(rankings, relevant, k):
    return np.mean([bool(set(r[:k]) & set(gold)) for r, gold in zip(rankings, relevant)])
def reciprocal_rank(rankings, relevant):
    vals = []
    for ranking, gold in zip(rankings, relevant):
        ranks = [i + 1 for i, doc_id in enumerate(ranking) if doc_id in gold]
        vals.append(1 / min(ranks) if ranks else 0)
    return float(np.mean(vals))

rankings = [list(np.argsort(-row)) for row in scores]
gold = [{0}, {1}]
print("recall@1", recall_at_k(rankings, gold, 1), "MRR", reciprocal_rank(rankings, gold))


## 28.2 Indexes and rerankers

Exact search scans every vector. Approximate nearest-neighbor indexes trade recall for
speed/memory using structures such as HNSW or inverted files. A cross-encoder jointly
reads query-document pairs and is slower but often more precise; retrieve many cheaply,
then rerank a smaller candidate set. Hybrid retrieval combines lexical and dense scores.
Always evaluate the entire retrieval cascade on labeled queries.


## 28.3 How embedding models are trained

Bi-encoders learn geometry from positive and negative pairs using contrastive losses. In-batch
negatives make other examples' documents serve as negatives; quality depends on batch size and
false-negative rate. Multiple Negatives Ranking Loss and InfoNCE-like objectives raise positive
similarity relative to negatives. Hard negatives—topically related but irrelevant documents—
teach fine distinctions, while accidental true positives mislabeled negative damage training.

Query/document encoders may share weights or use asymmetric prompts such as `query:` and
`passage:`. Some models require instruction prefixes. Pooling can use CLS, mean token pooling,
or learned mechanisms. Normalization changes the score from magnitude-sensitive dot product to
cosine. Matryoshka-trained embeddings support truncating dimensions with graceful degradation;
arbitrary dimension truncation does not. Follow the model card's encoding recipe exactly.


In [ ]:
# Inspect score distribution, margins, and retrieval confidence diagnostics.
for qi, query in enumerate(queries):
    order = np.argsort(-scores[qi])
    top = scores[qi, order[:3]]
    print(query)
    print(" top IDs:", order[:3].tolist(), "scores:", top.round(3).tolist(),
          "top1-top2 margin:", round(float(top[0] - top[1]), 3))
# Scores are model/index-specific; do not treat one universal threshold as confidence.


## 28.4 Lexical, dense, hybrid, and reranking systems

BM25 rewards query-term matches using term frequency, inverse document frequency, and length
normalization. It excels at identifiers, rare names, error codes, and exact wording. Dense
retrieval captures paraphrase and semantic relationships but can miss rare lexical details.
Hybrid retrieval retrieves from both, normalizes or rank-fuses results, then optionally
reranks. Reciprocal Rank Fusion combines ranks without assuming score comparability.

A cross-encoder jointly processes query and candidate, allowing full token interaction. It is
too expensive for the whole corpus but effective on top 20–200 candidates. Late-interaction
models retain token-level representations for a middle ground. The cascade's candidate count,
deduplication, metadata filters, reranker batch size, and final k are tuned against labeled
queries and latency. A better reranker cannot recover documents absent from candidate recall.


In [ ]:
# Reciprocal-rank fusion for heterogeneous retrievers.
def rrf(rankings, k=60):
    fused = {}
    for ranking in rankings:
        for rank, doc_id in enumerate(ranking, 1):
            fused[doc_id] = fused.get(doc_id, 0) + 1 / (k + rank)
    return sorted(fused, key=fused.get, reverse=True), fused

dense_rank = [0, 3, 1, 2]
lexical_rank = [3, 0, 2, 1]
fused_order, fused_scores = rrf([dense_rank, lexical_rank])
print(fused_order, {i: round(fused_scores[i], 4) for i in fused_order})


## 28.5 Index structures and evaluation details

Brute-force exact search is the correctness baseline. HNSW builds a navigable graph and offers
strong recall/latency with memory overhead. IVF partitions vectors into coarse cells and probes
selected lists. Product quantization compresses vectors with accuracy tradeoffs. Index choice
depends on corpus size, dimensionality, update pattern, filters, latency, memory, and target
recall. Benchmark on production hardware with representative filters and concurrent queries.

Recall@k asks whether any relevant document appears; precision@k measures returned relevance;
MRR rewards the first relevant rank; nDCG handles graded relevance and multiple relevant items.
Label incompleteness makes unjudged retrieved documents ambiguous, not automatically irrelevant.
Slice queries by intent, lexical rarity, language, freshness, answerability, and source. Track
embedding/index version and run reindex migrations explicitly. Mixing vectors from different
embedding revisions silently corrupts similarity.

**Operational checklist:** stable chunk IDs; normalized text policy; batch embedding; retry and
checksum; dimension/normalization validation; atomic index version swap; deletion propagation;
metadata ACL filters before exposure; and retrieval traces with sensitive content controls.


## 28.6 Semantic-search reference

| Stage | Primary metric/question |
|---|---|
| Encoder selection | Does its training recipe match query/document domain? |
| Exact baseline | What is best achievable recall for these embeddings? |
| ANN index | Recall loss versus latency/memory/build/update cost |
| Metadata filtering | Are ACL/tenant/time constraints correct? |
| Hybrid retrieval | Does lexical+dense improve slices? |
| Reranking | Does precision rise without losing required recall/latency? |

Cosine equals dot product only after unit normalization. Euclidean ranking of normalized vectors is
monotonically related, but index configuration must match. Similarity magnitudes are not calibrated
relevance probabilities and shift by model/domain/query length. Tune thresholds on labeled data and
include a no-result option.

Version embedding model/revision, preprocessing/prefix, dimension, normalization, distance function,
index parameters, corpus snapshot, and IDs. Never mix vector generations. Evaluate queries with
multiple relevant documents and incomplete judgments carefully; inspect failures qualitatively.


## 28.7 Pooling, normalization, and similarity

An embedding model's pooling and normalization are part of its contract. Mean pooling must exclude padding and divide by valid-token count; CLS pooling is valid only when trained for it. Cosine similarity equals a dot product only after L2 normalization. Query and document encoders may require different prefixes or prompts. Test batching invariance and empty inputs, and store model, tokenizer, pooling, normalization, dimensionality, and revision beside the index. Mixing embeddings from different revisions silently corrupts retrieval.


In [ ]:
hidden=torch.tensor([[[1.,0.],[0.,2.],[9.,9.]],[[2.,2.],[8.,8.],[8.,8.]]]); mask=torch.tensor([[1,1,0],[1,0,0]],dtype=torch.bool)
pooled=(hidden*mask.unsqueeze(-1)).sum(1)/mask.sum(1,keepdim=True); normalized=torch.nn.functional.normalize(pooled,dim=-1)
print(pooled,normalized,normalized@normalized.T)


## 28.8 Exact-search baseline before an ANN index

Approximate nearest-neighbor systems trade recall for memory and latency. Establish exact dot-product or cosine search on a manageable frozen corpus first, then compare index configurations against its top-k results. Report Recall@k of the ANN layer separately from semantic relevance. Filter and authorization constraints may require prefiltering, partitioned indexes, or oversampling followed by postfiltering. Rebuild or version indexes atomically when embeddings change, and retain document IDs rather than treating vector row numbers as durable identity.


In [ ]:
corpus=torch.nn.functional.normalize(torch.randn(100,8,generator=torch.Generator().manual_seed(2)),dim=-1); query=torch.nn.functional.normalize(torch.randn(8,generator=torch.Generator().manual_seed(3)),dim=0)
scores=corpus@query; top=torch.topk(scores,5); print(list(zip(top.indices.tolist(),top.values.tolist())))


## Primary references and further study

Use the pinned library documentation that matches your environment. Papers explain the method and assumptions; current official documentation defines the executable API.

- [Sentence-BERT](https://arxiv.org/abs/1908.10084)
- [FAISS](https://faiss.ai/)


## Exercises

    1. Create 20 labeled queries and compare lexical versus dense retrieval.
2. Add a cross-encoder reranker and measure recall/latency changes.
3. Demonstrate how normalization changes dot-product ranking.

    ## Checkpoint

    Explain the notebook's central mechanism without using library names, then identify
    one assumption you would test before applying it to a real workload.
